MODELO DE REGRESSÃO LOGÍSTICA — CHURN VOLUNTÁRIO

Seguradora Xperiun

Autor:    Bruno Cesar Pasquini (com assistência de IA)

Objetivo: Modelo explicativo de churn voluntário em apólices de seguros

# Etapa 2 - Importação e Validação dos Dados

## 2.1 - Importação de Bibliotecas

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import warnings
from boruta import BorutaPy
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from scipy import stats
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, brier_score_loss,
                             roc_curve)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")
print("✅ Bibliotecas importadas com sucesso. Ambiente pronto.")

✅ Bibliotecas importadas com sucesso. Ambiente pronto.


## 2.2 — Leitura do CSV exportado do SQLite

In [2]:
CAMINHO_CSV = 'C:\\Users\\bruno\\repos\\mba_xperiun_repos\\mba_xperiun_dc4\\Dados\\Tabela_para_Modelagem.csv'

df = pd.read_csv(
    CAMINHO_CSV
    , sep=';'            # ajuste se necessário (';' para CSVs brasileiros)
    , encoding='utf-8'   # ajuste se necessário ('latin-1' para acentos)
    , low_memory=False
)

print(f"Shape: {df.shape}")

# Verificação rápida de tipos e nulos (sanidade)
print(f"📦 Dataset carregado: {df.shape[0]} linhas, {df.shape[1]} colunas.")
print(f"🔹 Colunas numéricas: {df.select_dtypes(include=['number']).shape[1]}")
print(f"🔹 Colunas categóricas: {df.select_dtypes(include=['object']).shape[1]}")
print(f"🎯 Target (Churn): {df['indicador_churn'].sum()} ({df['indicador_churn'].mean()*100:.2f}%)\n")
print(f"💡 O modelo usará este valor dinâmico para correção de intercepto e thresholds.")

Shape: (97148, 32)
📦 Dataset carregado: 97148 linhas, 32 colunas.
🔹 Colunas numéricas: 21
🔹 Colunas categóricas: 11
🎯 Target (Churn): 17521 (18.04%)

💡 O modelo usará este valor dinâmico para correção de intercepto e thresholds.


## 2.3 - Validação Inicial

### 2.3.1 - Tipos de Dados

In [3]:
print("=" * 76)
print("Tipos de Dados")
print("=" * 76)
print(df.dtypes.to_string())

Tipos de Dados
apolice_id                    int64
cliente_id                    int64
produto_id                    int64
canal_id                      int64
data_inicio                     str
indicador_churn               int64
tipo_cliente                    str
genero                          str
faixa_etaria                    str
tempo_cliente_meses           int64
regiao                          str
ramo                            str
cobertura                       str
franquia_media              float64
nome_canal                      str
tipo_canal                      str
comissao_percentual         float64
vigencia_meses                int64
forma_pagamento                 str
ratio_premio_vs_medio       float64
mes_inicio                    int64
trimestre_inicio              int64
is_fim_semana                 int64
is_feriado                    int64
qtd_apolices_acumuladas       int64
qtd_ramos_distintos           int64
qtd_coberturas_distintas      int64
tem_auto     

### 2.3.2 - Valores Nulos

In [4]:
print("=" * 76)
print("Valores Nulos por Coluna")
print("=" * 76)
nulos = df.isnull().sum()
nulos_pct = (df.isnull().sum() / len(df) * 100).round(2)
resumo_nulos = pd.DataFrame({
    'nulos': nulos
    , 'pct': nulos_pct
})
print(resumo_nulos[resumo_nulos['nulos'] > 0].to_string())
if resumo_nulos['nulos'].sum() == 0:
    print("Nenhum valor nulo encontrado.")

Valores Nulos por Coluna
Empty DataFrame
Columns: [nulos, pct]
Index: []
Nenhum valor nulo encontrado.


### 2.3.3 - Taxa de Churn

In [5]:
taxa_churn = df['indicador_churn'].mean()
n_churn = df['indicador_churn'].sum()
n_nao_churn = len(df) - n_churn
print("=" * 76)
print("Distribuição da Variável Resposta")
print("=" * 76)
print(f"Total de registros: {len(df):,}")
print(f"Churn = 1:          {n_churn:,} ({taxa_churn:.2%})")
print(f"Churn = 0:          {n_nao_churn:,} ({1 - taxa_churn:.2%})")

# Validação: taxa deve estar próxima de 18%
assert 0.15 < taxa_churn < 0.21, \
    f"ALERTA: Taxa de churn ({taxa_churn:.2%}) fora do esperado (~18%)"
print(f"\n✅ Taxa de churn ({taxa_churn:.2%}) dentro do intervalo esperado.")

Distribuição da Variável Resposta
Total de registros: 97,148
Churn = 1:          17,521 (18.04%)
Churn = 0:          79,627 (81.96%)

✅ Taxa de churn (18.04%) dentro do intervalo esperado.


### 2.3.4 - Estatísticas Descritivas das Variáveis Contínuas

In [6]:
print("=" * 96)
print("Estatísticas Descritivas — Variáveis Contínuas")
print("=" * 96)
continuas = [
    'tempo_cliente_meses', 'premio_medio_mensal', 'franquia_media'
    , 'comissao_percentual', 'comissao', 'vigencia_meses', 'premio_mensal'
    , 'receita_esperada', 'ratio_premio_vs_medio'
    , 'qtd_apolices_acumuladas', 'qtd_ramos_distintos'
    , 'qtd_coberturas_distintas'
]
# Filtra apenas colunas que existam no DataFrame
continuas_existentes = [c for c in continuas if c in df.columns]
print(df[continuas_existentes].describe().round(2).transpose().to_string())

Estatísticas Descritivas — Variáveis Contínuas
                            count     mean      std   min    25%      50%      75%      max
tempo_cliente_meses       97148.0    31.01    23.48  1.00  10.00    26.00    49.00    84.00
franquia_media            97148.0  1616.68  1374.77  0.00   0.00  1500.00  2800.00  3500.00
comissao_percentual       97148.0     0.11     0.05  0.05   0.05     0.12     0.15     0.18
vigencia_meses            97148.0    13.20     4.89  6.00  12.00    12.00    12.00    24.00
ratio_premio_vs_medio     97148.0     1.14     0.19  0.90   1.01     1.09     1.21     1.94
qtd_apolices_acumuladas   97148.0     2.99     1.82  1.00   2.00     3.00     4.00    15.00
qtd_ramos_distintos       97148.0     1.81     0.74  1.00   1.00     2.00     2.00     3.00
qtd_coberturas_distintas  97148.0     1.86     0.75  1.00   1.00     2.00     2.00     3.00


### 2.3.5 - Distribuição das Variáveis Categóricas

In [7]:
print("=" * 76)
print("Distribuição — Variáveis Categóricas")
print("=" * 76)
categoricas = [
    'tipo_cliente', 'genero', 'faixa_etaria', 'regiao'
    , 'ramo', 'cobertura', 'nome_canal', 'tipo_canal'
    , 'forma_pagamento'
]
categoricas_existentes = [c for c in categoricas if c in df.columns]
for col in categoricas_existentes:
    print(f"\n--- {col} ---")
    contagem = df[col].value_counts()
    pct = (contagem / len(df) * 100).round(2)
    resumo = pd.DataFrame({'n': contagem, '%': pct})
    print(resumo.to_string())

Distribuição — Variáveis Categóricas

--- tipo_cliente ---
                     n      %
tipo_cliente                 
Pessoa Física    82653  85.08
Pessoa Jurídica  14495  14.92

--- genero ---
               n      %
genero                 
Masculino  50744  52.23
Feminino   46404  47.77

--- faixa_etaria ---
                  n      %
faixa_etaria              
3) 36-45      24741  25.47
2) 26-35      24523  25.24
4) 46-55      19301  19.87
5) 56-65      11548  11.89
1) 18-25       9258   9.53
6) 65+         7777   8.01

--- regiao ---
                  n      %
regiao                    
Sudeste       39525  40.69
Nordeste      26568  27.35
Sul           14889  15.33
Norte          8716   8.97
Centro-Oeste   7450   7.67

--- ramo ---
                 n     %
ramo                    
Automóvel    49352  50.8
Vida         29340  30.2
Residencial  18456  19.0

--- cobertura ---
               n      %
cobertura              
Completa   38802  39.94
Básica     36844  37.93
Premium    2

## 2.4 - Definição de papéis das colunas

In [8]:
# Colunas de identificação (não entram no modelo, mas são mantidas para rastreio)
COLS_ID = [
    'apolice_id', 'cliente_id', 'produto_id', 'canal_id', 'data_inicio'
]

# Variável resposta
COL_TARGET = 'indicador_churn'

# Variáveis usadas apenas para estratificação (não entram no modelo)
COLS_ESTRATIFICACAO = ['faixa_tempo_casa']

# Variáveis binárias que já são numéricas (não precisam de encoding)
COLS_BINARIAS = [
    'is_fim_semana', 'is_feriado'
    , 'tem_auto', 'tem_vida', 'tem_residencial', 'tem_cross_sell'
]

# Variáveis contínuas
COLS_CONTINUAS = [
    'tempo_cliente_meses', 'franquia_media'
    , 'comissao_percentual', 'vigencia_meses', 'ratio_premio_vs_medio'
    , 'qtd_apolices_acumuladas', 'qtd_ramos_distintos'
    , 'qtd_coberturas_distintas'
]

# Variáveis categóricas (precisarão de encoding)
COLS_CATEGORICAS = [
    'tipo_cliente', 'genero', 'faixa_etaria', 'regiao'
    , 'ramo', 'cobertura', 'nome_canal', 'tipo_canal'
    , 'forma_pagamento'
]

# Variáveis temporais que serão tratadas como categóricas
COLS_TEMPORAIS_CAT = ['mes_inicio', 'trimestre_inicio']

# ano_inicio: pode ser categórica ou contínua, vamos avaliar
# COLS_TEMPORAIS_CONT = ['ano_inicio']

# Features totais (tudo que é candidato ao modelo)
COLS_FEATURES = (
    COLS_CONTINUAS
    + COLS_BINARIAS
    + COLS_CATEGORICAS
    + COLS_TEMPORAIS_CAT
#     + COLS_TEMPORAIS_CONT  // Não acho que faça sentido incluir ano_inicio. Vamos deixar de fora por enquanto.
)

print(f"IDs:              {len(COLS_ID)}")
print(f"Target:           1 ({COL_TARGET})")
print(f"Contínuas:        {len(COLS_CONTINUAS)}")
print(f"Binárias:         {len(COLS_BINARIAS)}")
print(f"Categóricas:      {len(COLS_CATEGORICAS)}")
print(f"Temporais (cat):  {len(COLS_TEMPORAIS_CAT)}")
# print(f"Temporais (cont): {len(COLS_TEMPORAIS_CONT)}")  // Não acho que faça sentido incluir ano_inicio. Vamos deixar de fora por enquanto.
print(f"Estratificação:   {len(COLS_ESTRATIFICACAO)}")
print(f"Total features:   {len(COLS_FEATURES)}")

IDs:              5
Target:           1 (indicador_churn)
Contínuas:        8
Binárias:         6
Categóricas:      9
Temporais (cat):  2
Estratificação:   1
Total features:   25


## 2.5 - Conversão de Tipos

In [9]:
# Garante que categóricas sejam string e contínuas sejam float

for col in COLS_CATEGORICAS + COLS_TEMPORAIS_CAT:
    if col in df.columns:
        df[col] = df[col].astype(str)

# for col in COLS_CONTINUAS + COLS_TEMPORAIS_CONT:
for col in COLS_CONTINUAS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in COLS_BINARIAS:
    if col in df.columns:
        df[col] = df[col].astype(int)

print("✅ Tipos de dados convertidos.")
print(df[COLS_FEATURES].dtypes.to_string())

✅ Tipos de dados convertidos.
tempo_cliente_meses           int64
franquia_media              float64
comissao_percentual         float64
vigencia_meses                int64
ratio_premio_vs_medio       float64
qtd_apolices_acumuladas       int64
qtd_ramos_distintos           int64
qtd_coberturas_distintas      int64
is_fim_semana                 int64
is_feriado                    int64
tem_auto                      int64
tem_vida                      int64
tem_residencial               int64
tem_cross_sell                int64
tipo_cliente                    str
genero                          str
faixa_etaria                    str
regiao                          str
ramo                            str
cobertura                       str
nome_canal                      str
tipo_canal                      str
forma_pagamento                 str
mes_inicio                      str
trimestre_inicio                str


# Etapa 3 - Segmentação da Base (Desenvolvimento 85% / Teste 15%)

## 3.1 - Construção da chave de estratificação

Estratificamos por múltiplas variáveis usando iterstrat.
A biblioteca espera uma matriz binária (multi-label), então
convertemos cada variável de estratificação em colunas binárias.

### 3.1.1 - Bases de Treino e Teste

Variáveis de estratificação (por ordem de importância):
  01. indicador_churn
  02. faixa_tempo_casa
  03. cobertura
  04. tipo_canal
  05. ramo
  06. nome_canal
  07. tipo_cliente
  08. regiao
  09. genero
  10. faixa_etaria

In [10]:
# Variáveis que queremos preservar na estratificação
vars_estrat = [
    COL_TARGET         # indicador_churn
    , 'faixa_tempo_casa'
    , 'cobertura'
    , 'tipo_canal'
    , 'ramo'
    , 'nome_canal'
    , 'tipo_cliente'
    , 'regiao'
    , 'genero'
    , 'faixa_etaria'
]

# Cria matriz binária para cada variável de estratificação
# (cada categoria vira uma coluna 0/1)
estratos_df = pd.get_dummies(df[vars_estrat], columns=vars_estrat)
print(f"Matriz de estratificação: {estratos_df.shape}")
print(f"Colunas: {list(estratos_df.columns)}")

Matriz de estratificação: (97148, 36)
Colunas: ['indicador_churn_0', 'indicador_churn_1', 'faixa_tempo_casa_1) 0-12 meses', 'faixa_tempo_casa_2) 13-48 meses', 'faixa_tempo_casa_3) >48 meses', 'cobertura_Básica', 'cobertura_Completa', 'cobertura_Premium', 'tipo_canal_Corretor', 'tipo_canal_Digital', 'tipo_canal_Direto', 'tipo_canal_Parceria', 'ramo_Automóvel', 'ramo_Residencial', 'ramo_Vida', 'nome_canal_Bancassurance', 'nome_canal_Corretor Pessoa Física', 'nome_canal_Corretora Parceira', 'nome_canal_Televendas', 'nome_canal_Venda Digital (App)', 'nome_canal_Venda Digital (Site)', 'tipo_cliente_Pessoa Física', 'tipo_cliente_Pessoa Jurídica', 'regiao_Centro-Oeste', 'regiao_Nordeste', 'regiao_Norte', 'regiao_Sudeste', 'regiao_Sul', 'genero_Feminino', 'genero_Masculino', 'faixa_etaria_1) 18-25', 'faixa_etaria_2) 26-35', 'faixa_etaria_3) 36-45', 'faixa_etaria_4) 46-55', 'faixa_etaria_5) 56-65', 'faixa_etaria_6) 65+']


### 3.1.2 - Chave de Estratificação - Base de Modelagem

Variáveis de estratificação (por ordem de importância):
  01. faixa_tempo_casa
  02. cobertura
  03. tipo_canal
  04. ramo
  05. nome_canal
  06. tipo_cliente
  07. regiao
  08. genero
  09. faixa_etaria

In [11]:
# Variáveis que queremos preservar na estratificação
vars_estrat_model = [
      'faixa_tempo_casa'
    , 'cobertura'
    , 'tipo_canal'
    , 'ramo'
    , 'nome_canal'
    , 'tipo_cliente'
    , 'regiao'
    , 'genero'
    , 'faixa_etaria'
]

# Cria matriz binária para cada variável de estratificação
# (cada categoria vira uma coluna 0/1)
estratos_model_df = pd.get_dummies(df[vars_estrat_model], columns=vars_estrat_model)
print(f"Matriz de estratificação: {estratos_model_df.shape}")
print(f"Colunas: {list(estratos_model_df.columns)}")

Matriz de estratificação: (97148, 34)
Colunas: ['faixa_tempo_casa_1) 0-12 meses', 'faixa_tempo_casa_2) 13-48 meses', 'faixa_tempo_casa_3) >48 meses', 'cobertura_Básica', 'cobertura_Completa', 'cobertura_Premium', 'tipo_canal_Corretor', 'tipo_canal_Digital', 'tipo_canal_Direto', 'tipo_canal_Parceria', 'ramo_Automóvel', 'ramo_Residencial', 'ramo_Vida', 'nome_canal_Bancassurance', 'nome_canal_Corretor Pessoa Física', 'nome_canal_Corretora Parceira', 'nome_canal_Televendas', 'nome_canal_Venda Digital (App)', 'nome_canal_Venda Digital (Site)', 'tipo_cliente_Pessoa Física', 'tipo_cliente_Pessoa Jurídica', 'regiao_Centro-Oeste', 'regiao_Nordeste', 'regiao_Norte', 'regiao_Sudeste', 'regiao_Sul', 'genero_Feminino', 'genero_Masculino', 'faixa_etaria_1) 18-25', 'faixa_etaria_2) 26-35', 'faixa_etaria_3) 36-45', 'faixa_etaria_4) 46-55', 'faixa_etaria_5) 56-65', 'faixa_etaria_6) 65+']


## 3.2 - Split Estratificados (85/15 e 50/50)

### 3.2.1 - Split da Base Total 85/15

In [12]:
print("=" * 76)
print("3.2.1 — Split Estratificado 85/15")
print("=" * 76)

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.15, random_state=42
)

# X pode ser qualquer array do mesmo tamanho; o split usa apenas y (estratos)
X_placeholder = np.zeros((len(df), 1))
y_estratos = estratos_df.values

for train_idx, test_idx in msss.split(X_placeholder, y_estratos):
    idx_treino = train_idx
    idx_teste = test_idx

# Cria as bases
df_treino = df.iloc[idx_treino].copy().reset_index(drop=True)
df_teste  = df.iloc[idx_teste].copy().reset_index(drop=True)

print(f"\n{'=' * 76}")
print(f"Resultado do Split")
print(f"{'=' * 76}")
print(f"   Base completa: {len(df):>10,}")
print(f"📦 Treino:        {len(df_treino):>10,} ({len(df_treino)/len(df):.1%})")
print(f"📦 Teste:         {len(df_teste):>10,} ({len(df_teste)/len(df):.1%})")
print(f"🎯 Churn treino:  {df_treino['indicador_churn'].mean():.2%}")
print(f"🎯 Churn teste:   {df_teste['indicador_churn'].mean():.2%}")

3.2.1 — Split Estratificado 85/15

Resultado do Split
   Base completa:     97,148
📦 Treino:            82,575 (85.0%)
📦 Teste:             14,573 (15.0%)
🎯 Churn treino:  18.04%
🎯 Churn teste:   18.03%


### 3.2.2 - Undersample da Base de Treino para 50/50

In [13]:
print(f"\n{'=' * 76}")
print("3.2 — Undersampling Estratificado (50/50)")
print("=" * 76)

df_churn     = df_treino[df_treino['indicador_churn'] == 1].copy().reset_index(drop=True)
df_nao_churn = df_treino[df_treino['indicador_churn'] == 0].copy().reset_index(drop=True)

n_churn = len(df_churn)
n_nao_churn = len(df_nao_churn)
frac_manter = n_churn / n_nao_churn  # fração dos não-churns a manter

print(f"  Churns no treino:     {n_churn:,}")
print(f"  Não-churns no treino: {n_nao_churn:,}")
print(f"  Fração a manter:      {frac_manter:.4f}")

# Estratificação SEM indicador_churn (todos aqui são não-churn)
estratos_nao_churn = df_nao_churn[vars_estrat_model].copy()

# Se alguma coluna de estrato não existir em df_nao_churn, recrie a partir do df original
# Ajuste conforme suas colunas de estratificação
estratos_nc_values = estratos_nao_churn.values

msss_under = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=frac_manter, random_state=42
)

X_ph = np.zeros((len(df_nao_churn), 1))
for _, keep_idx in msss_under.split(X_ph, estratos_nc_values):
    idx_manter = keep_idx

df_nao_churn_amostrado = df_nao_churn.iloc[idx_manter].copy().reset_index(drop=True)

# Concatena: churn (100%) + não-churn (amostrado)
df_treino_bal = pd.concat([df_churn, df_nao_churn_amostrado], ignore_index=True)
df_treino_bal = df_treino_bal.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n⚖️ Undersampling aplicado.")
print(f"📊 Base do Modelo (Treino Balanceado): {len(df_treino_bal):,}")
print(f"   Churn:     {df_treino_bal['indicador_churn'].sum():,} ({df_treino_bal['indicador_churn'].mean():.2%})")
print(f"   Não-churn: {(df_treino_bal['indicador_churn'] == 0).sum():,} ({1 - df_treino_bal['indicador_churn'].mean():.2%})")


3.2 — Undersampling Estratificado (50/50)
  Churns no treino:     14,893
  Não-churns no treino: 67,682
  Fração a manter:      0.2200

⚖️ Undersampling aplicado.
📊 Base do Modelo (Treino Balanceado): 29,786
   Churn:     14,893 (50.00%)
   Não-churn: 14,893 (50.00%)


## 3.3 - Validação do Split: Proporções das Variáveis de Estratificação

### 3.3.1 - Proporções das Bases de Treino e Teste

In [14]:
print(f"\n{'=' * 76}")
print(f"Validação do Split — Proporções")
print(f"{'=' * 76}")

for var in vars_estrat:
    print(f"\n--- {var} ---")
    prop_total = (df[var].value_counts(normalize=True) * 100).round(2)
    prop_treino = (df_treino[var].value_counts(normalize=True) * 100).round(2)
    prop_teste = (df_teste[var].value_counts(normalize=True) * 100).round(2)
    comparacao = pd.DataFrame({
        'Total (%)': prop_total
        , 'Treino (%)': prop_treino
        , 'Teste (%)': prop_teste
    })
    comparacao['Diff (pp)'] = (comparacao['Treino (%)']
                               - comparacao['Teste (%)']).abs().round(2)
    print(comparacao.to_string())

# Validação da taxa de churn
churn_treino = df_treino[COL_TARGET].mean()
churn_teste = df_teste[COL_TARGET].mean()
print(f"\n--- Resumo Churn ---")
print(f"🎯 Churn treino: {churn_treino:.4%}")
print(f"🎯 Churn teste:  {churn_teste:.4%}")
print(f"   Diferença:    {abs(churn_treino - churn_teste):.4%}")


Validação do Split — Proporções

--- indicador_churn ---
                 Total (%)  Treino (%)  Teste (%)  Diff (pp)
indicador_churn                                             
0                    81.96       81.96      81.97       0.01
1                    18.04       18.04      18.03       0.01

--- faixa_tempo_casa ---
                  Total (%)  Treino (%)  Teste (%)  Diff (pp)
faixa_tempo_casa                                             
2) 13-48 meses        44.46       44.46      44.45       0.01
1) 0-12 meses         30.45       30.45      30.45       0.00
3) >48 meses          25.10       25.10      25.10       0.00

--- cobertura ---
           Total (%)  Treino (%)  Teste (%)  Diff (pp)
cobertura                                             
Completa       39.94       39.94      39.94        0.0
Básica         37.93       37.93      37.93        0.0
Premium        22.13       22.13      22.13        0.0

--- tipo_canal ---
            Total (%)  Treino (%)  Teste (%)  Di

### 3.3.2 - Proporções das Bases de Treino e de Modelagem

In [15]:
print(f"\n{'=' * 76}")
print(f"Validação do Split — Proporções")
print(f"{'=' * 76}")

for var in vars_estrat:
    print(f"\n--- {var} ---")
    prop_total = (df[var].value_counts(normalize=True) * 100).round(2)
    prop_treino = (df_treino[var].value_counts(normalize=True) * 100).round(2)
    prop_model = (df_treino_bal[var].value_counts(normalize=True) * 100).round(2)
    comparacao = pd.DataFrame({
        'Total (%)': prop_total
        , 'Treino (%)': prop_treino
        , 'Modelo (%)': prop_model
    })
    comparacao['Diff (pp)'] = (comparacao['Treino (%)']
                               - comparacao['Modelo (%)']).abs().round(2)
    print(comparacao.to_string())

# Validação da taxa de churn
churn_treino = df_treino[COL_TARGET].mean()
churn_model = df_treino_bal[COL_TARGET].mean()
print(f"\n--- Resumo Churn ---")
print(f"🎯 Churn treino: {churn_treino:.4%}")
print(f"🎯 Churn modelo: {churn_model:.4%}")
print(f"   Diferença:    {abs(churn_treino - churn_model):.4%}")


Validação do Split — Proporções

--- indicador_churn ---
                 Total (%)  Treino (%)  Modelo (%)  Diff (pp)
indicador_churn                                              
0                    81.96       81.96        50.0      31.96
1                    18.04       18.04        50.0      31.96

--- faixa_tempo_casa ---
                  Total (%)  Treino (%)  Modelo (%)  Diff (pp)
faixa_tempo_casa                                              
2) 13-48 meses        44.46       44.46       43.73       0.73
1) 0-12 meses         30.45       30.45       33.81       3.36
3) >48 meses          25.10       25.10       22.46       2.64

--- cobertura ---
           Total (%)  Treino (%)  Modelo (%)  Diff (pp)
cobertura                                              
Básica         37.93       37.93       40.63       2.70
Completa       39.94       39.94       39.76       0.18
Premium        22.13       22.13       19.61       2.52

--- tipo_canal ---
            Total (%)  Treino (%) 

## 3.4 — Salva as partes para uso nas próximas etapas

As variáveis COLS_ID, COL_TARGET, COLS_FEATURES e COLS_ESTRATIFICACAO
ficam disponíveis no escopo do notebook para as etapas seguintes.
Não salvamos em CSV intermediário — seguimos no mesmo notebook.

In [16]:
print(f"\n✅ Bases prontas para a Etapa 4 (Encoding).")
print(f"   df_treino: {df_treino.shape}")
print(f"   df_treino_bal: {df_treino_bal.shape}")
print(f"   df_teste:  {df_teste.shape}")
print(f"\n   Variáveis de features:        {len(COLS_FEATURES)}")
print(f"   Variáveis de identificação:   {len(COLS_ID)}")
print(f"   Variáveis de estratificação:  {len(COLS_ESTRATIFICACAO)}")


✅ Bases prontas para a Etapa 4 (Encoding).
   df_treino: (82575, 32)
   df_treino_bal: (29786, 32)
   df_teste:  (14573, 32)

   Variáveis de features:        25
   Variáveis de identificação:   5
   Variáveis de estratificação:  1


# Etapa 4 — Encoding de Variáveis Categóricas

Estratégia:
  1. Para cada variável categórica, calcular a taxa de churn por categoria
  2. Identificar a categoria com churn mais próximo da média global (~18%)
     como referência (baseline), desde que tenha volume razoável
  3. Agregar categorias com massa muito pequena (<2% da base)
  4. Criar dummies (drop_first via remoção manual da referência)
  5. Aplicar o MESMO encoding na base de teste (consistência)

IMPORTANTE: Todo o encoding é definido na base de TREINO e depois
aplicado identicamente na base de TESTE. Isso evita data leakage.

## 4.1 — Análise de churn por categoria (base de treino)

Para cada variável categórica, mostramos:
  - Contagem e percentual de cada categoria
  - Taxa de churn de cada categoria
  - Distância da taxa de churn à média global
  - Indicação de qual será a referência (baseline)

In [17]:
TAXA_CHURN_GLOBAL = df_treino_bal[COL_TARGET].mean()
print(f"Taxa de churn global (modelagem): {TAXA_CHURN_GLOBAL:.4%}")
print(f"{'=' * 76}")

# Variáveis categóricas que precisam de encoding
# (inclui as temporais tratadas como categóricas)
TODAS_CATEGORICAS = COLS_CATEGORICAS + COLS_TEMPORAIS_CAT

# Dicionário para armazenar as decisões de encoding
encoding_config = {}

# Limiar mínimo de volume para uma categoria não ser agregada
LIMIAR_VOLUME_PCT = 5.0  # 5% da base de treino

for col in TODAS_CATEGORICAS:
    print(f"\n{'─' * 76}")
    print(f"VARIÁVEL: {col}")
    print(f"{'─' * 76}")

    # Contagem e taxa de churn por categoria
    analise = df_treino_bal.groupby(col).agg(
        n=(COL_TARGET, 'count')
        , churn_sum=(COL_TARGET, 'sum')
    ).reset_index()
    analise['pct_base'] = (analise['n'] / len(df_treino_bal) * 100).round(2)
    analise['taxa_churn'] = (analise['churn_sum'] / analise['n']).round(4)
    analise['dist_media'] = (
        (analise['taxa_churn'] - TAXA_CHURN_GLOBAL).abs()
    ).round(4)

    # Ordena por distância à média global
    analise = analise.sort_values('dist_media').reset_index(drop=True)

    # Identifica categorias pequenas (< LIMIAR_VOLUME_PCT)
    analise['pequena'] = analise['pct_base'] < LIMIAR_VOLUME_PCT

    # Seleciona a referência:
    # - Churn mais próximo da média global
    # - Que NÃO seja uma categoria pequena
    candidatas_ref = analise[~analise['pequena']]
    if len(candidatas_ref) > 0:
        referencia = candidatas_ref.iloc[0][col]
    else:
        # Se todas forem pequenas, usa a maior
        referencia = analise.sort_values('n', ascending=False).iloc[0][col]

    analise['papel'] = analise[col].apply(
        lambda x: '→ REFERÊNCIA' if x == referencia else ''
    )

    # Marca categorias pequenas
    analise.loc[analise['pequena'], 'papel'] = analise.loc[
        analise['pequena'], 'papel'
    ].apply(lambda x: '⚠️ PEQUENA' if x == '' else x)

    print(analise[[col, 'n', 'pct_base', 'taxa_churn', 'dist_media', 'papel']]
          .to_string(index=False))

    # Armazena configuração
    categorias_pequenas = list(analise[analise['pequena']][col])
    encoding_config[col] = {
        'referencia': referencia
        , 'categorias_pequenas': categorias_pequenas
        , 'todas_categorias': list(analise[col])
    }

    print(f"\n  Referência escolhida: '{referencia}' "
          f"(churn={analise[analise[col]==referencia]['taxa_churn'].values[0]:.2%}"
          f", dist={analise[analise[col]==referencia]['dist_media'].values[0]:.4f})")
    if categorias_pequenas:
        print(f"  ⚠️ Categorias pequenas (<{LIMIAR_VOLUME_PCT}%): "
              f"{categorias_pequenas}")

Taxa de churn global (modelagem): 50.0000%

────────────────────────────────────────────────────────────────────────────
VARIÁVEL: tipo_cliente
────────────────────────────────────────────────────────────────────────────
   tipo_cliente     n  pct_base  taxa_churn  dist_media        papel
  Pessoa Física 25300     84.94      0.4984      0.0016 → REFERÊNCIA
Pessoa Jurídica  4486     15.06      0.5089      0.0089             

  Referência escolhida: 'Pessoa Física' (churn=49.84%, dist=0.0016)

────────────────────────────────────────────────────────────────────────────
VARIÁVEL: genero
────────────────────────────────────────────────────────────────────────────
   genero     n  pct_base  taxa_churn  dist_media        papel
Masculino 15561     52.24      0.4984      0.0016 → REFERÊNCIA
 Feminino 14225     47.76      0.5017      0.0017             

  Referência escolhida: 'Masculino' (churn=49.84%, dist=0.0016)

────────────────────────────────────────────────────────────────────────────

## 4.2 — Decisão sobre agregação de categorias pequenas

Revisamos as categorias pequenas e decidimos como tratá-las.
Opções: agregar em "Outros", manter como estão, ou absorver na referência.

IMPORTANTE: Esta célula mostra o resumo para decisão. Se quiser alterar
alguma decisão, modifique o dicionário 'agregacoes' abaixo.

In [18]:
print(f"\n{'=' * 76}")
print("Resumo de Categorias Pequenas")
print(f"{'=' * 76}")

tem_pequenas = False
for col, config in encoding_config.items():
    if config['categorias_pequenas']:
        tem_pequenas = True
        print(f"\n  {col}: {config['categorias_pequenas']}")

if not tem_pequenas:
    print("\n  Nenhuma categoria com volume < 5%. Nenhuma agregação necessária.")

# Dicionário de agregações manuais.
# Formato: {'coluna': {'categoria_original': 'categoria_destino'}}
# Se nenhuma agregação for necessária, deixe vazio.
# Exemplo: {'regiao': {'Norte': 'Norte/Centro-Oeste', 'Centro-Oeste': 'Norte/Centro-Oeste'}}
agregacoes = {}


Resumo de Categorias Pequenas

  Nenhuma categoria com volume < 5%. Nenhuma agregação necessária.


## 4.3 — Aplicação das agregações (se houver)

In [19]:
def aplicar_agregacoes(dataframe, agregacoes_dict):
    """Aplica agregações de categorias em um DataFrame."""
    df_out = dataframe.copy()
    for col, mapeamento in agregacoes_dict.items():
        if col in df_out.columns:
            df_out[col] = df_out[col].replace(mapeamento)
            print(f"  Agregação aplicada em '{col}': {mapeamento}")
    return df_out

if agregacoes:
    print("Aplicando agregações na base de treino...")
    df_treino = aplicar_agregacoes(df_treino, agregacoes)
    print("Aplicando agregações na base de modelagem...")
    df_treino_bal = aplicar_agregacoes(df_treino_bal, agregacoes)
    print("\nAplicando agregações na base de teste...")
    df_teste = aplicar_agregacoes(df_teste, agregacoes)
    print("\n✅ Agregações aplicadas.")
else:
    print("Nenhuma agregação definida. Prosseguindo sem alterações.")

Nenhuma agregação definida. Prosseguindo sem alterações.


## 4.4 — Criação das variáveis dummy (One-Hot Encoding)

Para cada variável categórica:
  - Cria dummies para todas as categorias EXCETO a referência
  - Usa o prefixo da variável original para nomenclatura clara
  - Registra o mapeamento para documentação

As dummies são criadas na base de TREINO e o mesmo esquema é aplicado
na base de TESTE.

In [20]:
def criar_dummies(dataframe, col, referencia, prefixo=None):
    """
    Cria variáveis dummy para uma coluna categórica,
    removendo a categoria de referência.

    Retorna:
        DataFrame com as novas colunas dummy
        Lista com os nomes das colunas criadas
    """
    if prefixo is None:
        prefixo = col

    dummies = pd.get_dummies(
        dataframe[col]
        , prefix=prefixo
        , dtype=int
    )

    # Remove a coluna da referência
    col_referencia = f"{prefixo}_{referencia}"
    if col_referencia in dummies.columns:
        dummies = dummies.drop(columns=[col_referencia])
    else:
        # Tenta match parcial (por segurança com espaços/acentos)
        for c in dummies.columns:
            if referencia in c:
                dummies = dummies.drop(columns=[c])
                break

    return dummies, list(dummies.columns)


# Armazena o mapeamento completo de dummies criadas
mapeamento_dummies = {}
colunas_dummy_treino = []

print(f"{'=' * 76}")
print("Criação de Dummies — Base de Treino")
print(f"{'=' * 76}")

for col in TODAS_CATEGORICAS:
    referencia = encoding_config[col]['referencia']
    dummies_treino, nomes_colunas = criar_dummies(
        df_treino, col, referencia
    )

    # Adiciona as dummies ao DataFrame de treino
    df_treino = pd.concat([df_treino, dummies_treino], axis=1)
    colunas_dummy_treino.extend(nomes_colunas)
    mapeamento_dummies[col] = {
        'referencia': referencia
        , 'dummies_criadas': nomes_colunas
    }

    print(f"\n  {col}:")
    print(f"    Referência (baseline): '{referencia}'")
    print(f"    Dummies criadas ({len(nomes_colunas)}): {nomes_colunas}")

print(f"\n{'─' * 76}")
print(f"Total de variáveis dummy criadas: {len(colunas_dummy_treino)}")

Criação de Dummies — Base de Treino

  tipo_cliente:
    Referência (baseline): 'Pessoa Física'
    Dummies criadas (1): ['tipo_cliente_Pessoa Jurídica']

  genero:
    Referência (baseline): 'Masculino'
    Dummies criadas (1): ['genero_Feminino']

  faixa_etaria:
    Referência (baseline): '2) 26-35'
    Dummies criadas (5): ['faixa_etaria_1) 18-25', 'faixa_etaria_3) 36-45', 'faixa_etaria_4) 46-55', 'faixa_etaria_5) 56-65', 'faixa_etaria_6) 65+']

  regiao:
    Referência (baseline): 'Sul'
    Dummies criadas (4): ['regiao_Centro-Oeste', 'regiao_Nordeste', 'regiao_Norte', 'regiao_Sudeste']

  ramo:
    Referência (baseline): 'Residencial'
    Dummies criadas (2): ['ramo_Automóvel', 'ramo_Vida']

  cobertura:
    Referência (baseline): 'Completa'
    Dummies criadas (2): ['cobertura_Básica', 'cobertura_Premium']

  nome_canal:
    Referência (baseline): 'Bancassurance'
    Dummies criadas (5): ['nome_canal_Corretor Pessoa Física', 'nome_canal_Corretora Parceira', 'nome_canal_Televenda

## 4.5 — Aplicação do mesmo encoding em todas as bases

CRÍTICO: Usamos exatamente as mesmas colunas e referências definidas no treino. Se uma categoria aparecer no teste ou no modelo mas não no treino, ela será ignorada. Se uma dummy do treino não existir no teste, será preenchida com 0.

### 4.5.1 - Aplicação do mesmo encoding na Base de Teste

In [21]:
print(f"{'=' * 76}")
print("Aplicação do Encoding — Base de Teste")
print(f"{'=' * 76}")

colunas_dummy_teste = []

for col in TODAS_CATEGORICAS:
    referencia = encoding_config[col]['referencia']
    dummies_teste, nomes_teste = criar_dummies(
        df_teste, col, referencia
    )

    # Adiciona as dummies ao DataFrame de teste
    df_teste = pd.concat([df_teste, dummies_teste], axis=1)
    colunas_dummy_teste.extend(nomes_teste)

# Garante que treino e teste tenham exatamente as mesmas colunas dummy
colunas_faltando_teste = set(colunas_dummy_treino) - set(colunas_dummy_teste)
colunas_extras_teste = set(colunas_dummy_teste) - set(colunas_dummy_treino)

if colunas_faltando_teste:
    print(f"\n  ⚠️ Colunas do treino ausentes no teste (preenchidas com 0):")
    for c in colunas_faltando_teste:
        df_teste[c] = 0
        print(f"    - {c}")

if colunas_extras_teste:
    print(f"\n  ⚠️ Colunas extras no teste (removidas):")
    for c in colunas_extras_teste:
        df_teste = df_teste.drop(columns=[c])
        print(f"    - {c}")

if not colunas_faltando_teste and not colunas_extras_teste:
    print("\n  ✅ Colunas dummy idênticas entre treino e teste.")

Aplicação do Encoding — Base de Teste

  ✅ Colunas dummy idênticas entre treino e teste.


### 4.5.2 - Aplicação do mesmo encoding na Base de Modelagem

In [22]:
print(f"{'=' * 76}")
print("Aplicação do Encoding — Base de Modelagem")
print(f"{'=' * 76}")

colunas_dummy_model = []

for col in TODAS_CATEGORICAS:
    referencia = encoding_config[col]['referencia']
    dummies_model, nomes_model = criar_dummies(
        df_treino_bal, col, referencia
    )

    # Adiciona as dummies ao DataFrame de teste
    df_treino_bal = pd.concat([df_treino_bal, dummies_model], axis=1)
    colunas_dummy_model.extend(nomes_model)

# Garante que treino e model tenham exatamente as mesmas colunas dummy
colunas_faltando_model = set(colunas_dummy_treino) - set(colunas_dummy_model)
colunas_extras_model = set(colunas_dummy_model) - set(colunas_dummy_treino)

if colunas_faltando_teste:
    print(f"\n  ⚠️ Colunas do treino ausentes no modelo (preenchidas com 0):")
    for c in colunas_faltando_model:
        df_teste[c] = 0
        print(f"    - {c}")

if colunas_extras_model:
    print(f"\n  ⚠️ Colunas extras no modelo (removidas):")
    for c in colunas_extras_model:
        df_treino_bal = df_treino_bal.drop(columns=[c])
        print(f"    - {c}")

if not colunas_faltando_model and not colunas_extras_model:
    print("\n  ✅ Colunas dummy idênticas entre treino e modelo.")

Aplicação do Encoding — Base de Modelagem

  ✅ Colunas dummy idênticas entre treino e modelo.


## 4.6 — Tendo a certeza de não termos duplicado colunas

In [23]:
# ── Fix: remover colunas duplicadas no df_treino_bal ──
print(f"Colunas antes da dedup (base de treino): {df_treino.shape[1]}")
df_treino = df_treino.loc[:, ~df_treino.columns.duplicated()]
print(f"Colunas após dedup (base de treino):     {df_treino.shape[1]}")

print(f"Colunas antes da dedup (base de modelagem): {df_treino_bal.shape[1]}")
df_treino_bal = df_treino_bal.loc[:, ~df_treino_bal.columns.duplicated()]
print(f"Colunas após dedup (base de modelagem):     {df_treino_bal.shape[1]}")

print(f"Colunas antes da dedup (base de teste): {df_teste.shape[1]}")
df_teste = df_teste.loc[:, ~df_teste.columns.duplicated()]
print(f"Colunas após dedup (base de teste):     {df_teste.shape[1]}")

# Verificação
n_indicador = (df_treino_bal.columns == COL_TARGET).sum()
print(f"Ocorrências de '{COL_TARGET}': {n_indicador}")
assert n_indicador == 1, f"ERRO: {COL_TARGET} aparece {n_indicador} vezes!"

Colunas antes da dedup (base de treino): 72
Colunas após dedup (base de treino):     72
Colunas antes da dedup (base de modelagem): 72
Colunas após dedup (base de modelagem):     72
Colunas antes da dedup (base de teste): 72
Colunas após dedup (base de teste):     72
Ocorrências de 'indicador_churn': 1


## 4.7 — Atualização da lista de features do modelo

Substitui as variáveis categóricas originais pelas dummies.
As variáveis categóricas originais são mantidas no DataFrame
(para rastreio), mas NÃO entram no modelo.

In [24]:
# Features finais para o modelo:
#   Contínuas + Binárias originais + Temporais contínuas + Dummies
COLS_MODELO = (
    COLS_CONTINUAS
    + COLS_BINARIAS
#     + COLS_TEMPORAIS_CONT
    + colunas_dummy_model
)

print(f"\n{'=' * 76}")
print("Features Finais para o Modelo")
print(f"{'=' * 76}")
print(f"\n  Contínuas originais:    {len(COLS_CONTINUAS)}")
print(f"  Binárias originais:     {len(COLS_BINARIAS)}")
# print(f"  Temporais contínuas:    {len(COLS_TEMPORAIS_CONT)}")
print(f"  Dummies (encoding):     {len(colunas_dummy_model)}")
print(f"  {'─' * 40}")
print(f"  TOTAL FEATURES:         {len(COLS_MODELO)}")

print(f"\n  Lista completa:")
for i, col in enumerate(COLS_MODELO, 1):
#     tipo = 'CONT' if col in COLS_CONTINUAS + COLS_TEMPORAIS_CONT \
    tipo = 'CONT' if col in COLS_CONTINUAS \
        else 'BIN' if col in COLS_BINARIAS \
        else 'DUMMY'
    print(f"    {i:3d}. {col:<45s} [{tipo}]")


Features Finais para o Modelo

  Contínuas originais:    8
  Binárias originais:     6
  Dummies (encoding):     40
  ────────────────────────────────────────
  TOTAL FEATURES:         54

  Lista completa:
      1. tempo_cliente_meses                           [CONT]
      2. franquia_media                                [CONT]
      3. comissao_percentual                           [CONT]
      4. vigencia_meses                                [CONT]
      5. ratio_premio_vs_medio                         [CONT]
      6. qtd_apolices_acumuladas                       [CONT]
      7. qtd_ramos_distintos                           [CONT]
      8. qtd_coberturas_distintas                      [CONT]
      9. is_fim_semana                                 [BIN]
     10. is_feriado                                    [BIN]
     11. tem_auto                                      [BIN]
     12. tem_vida                                      [BIN]
     13. tem_residencial                            

## 4.8 — Validação final do encoding

In [25]:
print(f"\n{'=' * 76}")
print("Validação Final do Encoding")
print(f"{'=' * 76}")

# Verifica se todas as features existem nos dois DataFrames
features_ok = True
for col in COLS_MODELO:
    if col not in df_treino.columns:
        print(f"  ❌ Coluna '{col}' não encontrada no df_treino")
        features_ok = False
    if col not in df_treino_bal.columns:
        print(f"  ❌ Coluna '{col}' não encontrada no df_treino_bal")
        features_ok = False
    if col not in df_teste.columns:
        print(f"  ❌ Coluna '{col}' não encontrada no df_teste")
        features_ok = False

if features_ok:
    print(f"  ✅ Todas as {len(COLS_MODELO)} features presentes em treino, model e teste.")

# Verifica se não há NaN nas features
nan_treino = df_treino[COLS_MODELO].isnull().sum()
nan_model = df_treino_bal[COLS_MODELO].isnull().sum()
nan_teste = df_teste[COLS_MODELO].isnull().sum()

if nan_treino.sum() > 0:
    print(f"\n  ⚠️ NaN encontrados no treino:")
    print(nan_treino[nan_treino > 0].to_string())
else:
    print(f"  ✅ Sem valores nulos nas features do treino.")

if nan_model.sum() > 0:
    print(f"\n  ⚠️ NaN encontrados no modelo:")
    print(nan_model[nan_model > 0].to_string())
else:
    print(f"  ✅ Sem valores nulos nas features do modelo.")

if nan_teste.sum() > 0:
    print(f"\n  ⚠️ NaN encontrados no teste:")
    print(nan_teste[nan_teste > 0].to_string())
else:
    print(f"  ✅ Sem valores nulos nas features do teste.")

# Shape final
print(f"\n  df_treino: {df_treino.shape}")
print(f"  df_treino_bal:  {df_treino_bal.shape}")
print(f"  df_teste:  {df_teste.shape}")
print(f"  Features para modelo: {len(COLS_MODELO)}")
print(f"  Target: {COL_TARGET}")


Validação Final do Encoding
  ✅ Todas as 54 features presentes em treino, model e teste.
  ✅ Sem valores nulos nas features do treino.
  ✅ Sem valores nulos nas features do modelo.
  ✅ Sem valores nulos nas features do teste.

  df_treino: (82575, 72)
  df_treino_bal:  (29786, 72)
  df_teste:  (14573, 72)
  Features para modelo: 54
  Target: indicador_churn


## 4.9 — Documentação do encoding (para referência e exportação)

In [26]:
print(f"\n{'=' * 76}")
print("Documentação do Encoding — Referências (Baselines)")
print(f"{'=' * 76}")
print(f"\nTaxa de churn global (modelagem): {TAXA_CHURN_GLOBAL:.4%}")
print(f"\n{'─' * 76}")
print(f"{'Variável':<25s} {'Referência (baseline)':<35s} {'Churn Ref':>10s}")
print(f"{'─' * 76}")

for col, config in mapeamento_dummies.items():
    ref = config['referencia']
    # Calcula o churn da referência na base de modelagem (df_treino_bal) para ser mais alinhado ao modelo final
    mask = df_treino_bal[col] == ref
    if mask.sum() > 0:
        churn_ref = df_treino_bal.loc[mask, COL_TARGET].mean()
    else:
        churn_ref = float('nan')
    print(f"  {col:<25s} {ref:<35s} {churn_ref:>9.2%}")

print(f"{'─' * 76}")
print(f"\nInterpretação: Os coeficientes (betas) e odds ratios do modelo")
print(f"serão relativos a estas categorias de referência.")
print(f"Ex: Se beta(ramo_Vida) = 0.3, significa que 'Vida' tem")
print(f"    odds de churn 35% maiores que a referência ('{mapeamento_dummies.get('ramo', {}).get('referencia', 'N/A')}').")

print(f"\n✅ Etapa 4 concluída. Pronto para Etapa 5 (Seleção de Variáveis).")


Documentação do Encoding — Referências (Baselines)

Taxa de churn global (modelagem): 50.0000%

────────────────────────────────────────────────────────────────────────────
Variável                  Referência (baseline)                Churn Ref
────────────────────────────────────────────────────────────────────────────
  tipo_cliente              Pessoa Física                          49.84%
  genero                    Masculino                              49.84%
  faixa_etaria              2) 26-35                               49.59%
  regiao                    Sul                                    50.37%
  ramo                      Residencial                            49.81%
  cobertura                 Completa                               49.89%
  nome_canal                Bancassurance                          49.79%
  tipo_canal                Parceria                               49.79%
  forma_pagamento           Boleto                                 49.93%
  mes_inic

# Etapa 5 — Seleção de Variáveis

Métodos aplicados (todos na base de TREINO):

  5.0 — Preparação: padronização para métodos que exigem escala

  5.1 — VIF (multicolinearidade)

  5.2 — Análise bivariada (correlação + teste estatístico)

  5.3 — Lasso (L1 regularization)

  5.4 — AIC/BIC (critério de informação)

  5.5 — Boruta (wrapper com Random Forest)

  5.6 — Stepwise Bidirecional (Forward-First)

  5.7 — Stepwise Bidirecional (Backward-First)

  5.8 — Matriz de consenso

  5.9 — Seleção final de variáveis

## 5.0 — Preparação: padronização para métodos que exigem escala

Lasso, VIF e Boruta se beneficiam de variáveis na mesma escala.

Padronizamos as contínuas (z-score) apenas para seleção.

O modelo final (Etapa 7) usará os dados originais com padronização própria.

In [27]:
print("🚀 Iniciando pipeline de seleção de variáveis...\n")

# Separa X e y de treino
X_treino = df_treino_bal[COLS_MODELO].copy()
y_treino = df_treino_bal[COL_TARGET].copy()

# Identifica colunas contínuas (que precisam de padronização)
cols_para_padronizar = COLS_CONTINUAS # + COLS_TEMPORAIS_CONT

# Padroniza apenas as contínuas
scaler_selecao = StandardScaler()
X_treino_scaled = X_treino.copy()
X_treino_scaled[cols_para_padronizar] = scaler_selecao.fit_transform(
    X_treino[cols_para_padronizar]
)

print(f"X_treino shape: {X_treino.shape}")
print(f"y_treino shape: {y_treino.shape}")
print(f"Colunas padronizadas: {len(cols_para_padronizar)}")
print(f"Colunas não padronizadas (binárias/dummies): "
      f"{len(COLS_MODELO) - len(cols_para_padronizar)}")

🚀 Iniciando pipeline de seleção de variáveis...

X_treino shape: (29786, 54)
y_treino shape: (29786,)
Colunas padronizadas: 8
Colunas não padronizadas (binárias/dummies): 46


## 5.1 — VIF (Variance Inflation Factor)

 O VIF mede a multicolinearidade de cada feature com todas as demais.
   - VIF = 1: sem colinearidade
   - VIF 1-5: baixa (aceitável)
   - VIF 5-10: moderada (atenção)
   - VIF > 10: alta (problemática)

NOTA: Dummies do mesmo grupo categórico terão VIF elevado naturalmente (pois somam 1 com a referência omitida). Isso é esperado e tolerável. O foco é detectar colinearidade ENTRE grupos diferentes de variáveis.

In [28]:
print(f"\n{'=' * 76}")
print("5.1 — VIF (Multicolinearidade)")
print(f"{'=' * 76}")

# Calcula VIF usando dados padronizados (escala uniforme)
# Adiciona constante para o cálculo
X_vif = X_treino_scaled[COLS_MODELO].copy()
X_vif = X_vif.assign(const=1)

vif_data = []
print("\nCalculando VIF (pode levar alguns segundos)...")
for i, col in enumerate(COLS_MODELO):
    vif_val = variance_inflation_factor(X_vif.values, i)
    vif_data.append({'feature': col, 'VIF': round(vif_val, 2)})

df_vif = pd.DataFrame(vif_data).sort_values('VIF', ascending=False).reset_index(drop=True)

# Classificação
def classificar_vif(v):
    if v > 10: return '🔴 Alta'
    elif v > 5: return '🟡 Moderada'
    else: return '🟢 Baixa'

df_vif['classificacao'] = df_vif['VIF'].apply(classificar_vif)

# Identifica grupo categórico de cada dummy para contextualizar
def grupo_categorico(feature):
    for col_orig in TODAS_CATEGORICAS:
        if feature.startswith(col_orig + '_'):
            return col_orig
    return '—'

df_vif['grupo'] = df_vif['feature'].apply(grupo_categorico)

print(f"\n{'─' * 80}")
print(f"{'#':>3s} {'Feature':<45s} {'VIF':>8s} {'Class.':<15s} {'Grupo':<20s}")
print(f"{'─' * 80}")
for i, row in df_vif.iterrows():
    print(f"{df_vif.index.get_loc(i)+1:3d} {row['feature']:<45s} "
          f"{row['VIF']:>8.2f} {row['classificacao']:<15s} {row['grupo']:<20s}")

# Resumo
vif_alta = set(df_vif[df_vif['VIF'] > 10]['feature'])
vif_moderada = set(df_vif[(df_vif['VIF'] > 5) & (df_vif['VIF'] <= 10)]['feature'])
vif_baixa = set(df_vif[df_vif['VIF'] <= 5]['feature'])
print(f"\n  VIF > 10 (alta colinearidade):    {len(vif_alta)} features")
print(f"  VIF 5-10 (moderada):              {len(vif_moderada)} features")
print(f"  VIF <= 5 (aceitável):             {len(vif_baixa)} features")

# Identifica pares problemáticos (entre grupos diferentes)
if vif_alta:
    print(f"\n  ⚠️ Features com VIF > 10:")
    for feat in vif_alta:
        grupo = grupo_categorico(feat)
        print(f"    - {feat} (VIF={df_vif[df_vif['feature']==feat]['VIF'].values[0]:.1f})"
              f" [Grupo: {grupo}]")


5.1 — VIF (Multicolinearidade)

Calculando VIF (pode levar alguns segundos)...

────────────────────────────────────────────────────────────────────────────────
  # Feature                                            VIF Class.          Grupo               
────────────────────────────────────────────────────────────────────────────────
  1 comissao_percentual                                inf 🔴 Alta          —                   
  2 mes_inicio_9                                       inf 🔴 Alta          mes_inicio          
  3 trimestre_inicio_1                                 inf 🔴 Alta          trimestre_inicio    
  4 trimestre_inicio_3                                 inf 🔴 Alta          trimestre_inicio    
  5 trimestre_inicio_2                                 inf 🔴 Alta          trimestre_inicio    
  6 mes_inicio_10                                      inf 🔴 Alta          mes_inicio          
  7 mes_inicio_1                                       inf 🔴 Alta          mes_inicio

## 5.2 — Análise Bivariada

 Para cada feature, testamos a associação com o target (indicador_churn).
   - Contínuas: correlação ponto-bisserial + teste Mann-Whitney U
   - Binárias/Dummies: teste Qui-Quadrado + coeficiente Phi/Cramér's V

 Resultado: ranking por força de associação com o churn.

In [29]:
print(f"{'=' * 76}")
print("5.2 — Análise Bivariada: Associação com Churn")
print(f"{'=' * 76}")

resultados_bivariada = []

def safe_nunique(series):
    n = series.nunique()
    if isinstance(n, (pd.Series, np.ndarray)):
        return int(n.iloc[0]) if len(n) > 0 else 0
    return int(n)

for col in COLS_MODELO:
    valores = X_treino[col]
    n_unicos = safe_nunique(valores)
    if n_unicos <= 2:
        # Variável binária/dummy → Qui-Quadrado + Phi
        tabela = pd.crosstab(valores, y_treino)
        chi2, p_value, dof, expected = stats.chi2_contingency(tabela)
        n = len(valores)
        phi = np.sqrt(chi2 / n)
        resultados_bivariada.append({
            'feature': col
            , 'tipo_teste': 'Qui-Quadrado'
            , 'estatistica': chi2
            , 'p_value': p_value
            , 'efeito': phi
            , 'metrica_efeito': 'Phi'
        })
    else:
        # Variável contínua → Point-Biserial + Mann-Whitney
        grupo_0 = valores[y_treino == 0]
        grupo_1 = valores[y_treino == 1]
        # Point-biserial correlation
        r_pb, p_pb = stats.pointbiserialr(y_treino, valores)
        # Mann-Whitney U (não-paramétrico)
        u_stat, p_mw = stats.mannwhitneyu(
            grupo_0, grupo_1, alternative='two-sided'
        )
        resultados_bivariada.append({
            'feature': col
            , 'tipo_teste': 'Mann-Whitney'
            , 'estatistica': u_stat
            , 'p_value': p_mw
            , 'efeito': abs(r_pb)
            , 'metrica_efeito': '|r_pb|'
        })

df_bivariada = pd.DataFrame(resultados_bivariada)
df_bivariada = df_bivariada.sort_values('efeito', ascending=False).reset_index(drop=True)

# Significância estatística (Bonferroni: alpha = 0.05 / n_features)
alpha_bonferroni = 0.05 / len(COLS_MODELO)
df_bivariada['significativa'] = df_bivariada['p_value'] < alpha_bonferroni

# Classificação do tamanho do efeito
def classificar_efeito(row):
    e = row['efeito']
    if row['metrica_efeito'] == 'Phi':
        if e >= 0.3: return 'Grande'
        elif e >= 0.1: return 'Médio'
        else: return 'Pequeno'
    else:  # |r_pb|
        if e >= 0.3: return 'Grande'
        elif e >= 0.1: return 'Médio'
        else: return 'Pequeno'

df_bivariada['magnitude'] = df_bivariada.apply(classificar_efeito, axis=1)

# Exibe resultados
print(f"\nCorreção de Bonferroni: alpha = {alpha_bonferroni:.6f}")
print(f"\n{'─' * 110}")
print(f"{'#':>3s} {'Feature':<45s} {'Teste':<14s} {'Efeito':>8s} "
      f"{'Mag.':<10s} {'p-value':>12s} {'Sig.':>5s}")
print(f"{'─' * 110}")
for i, row in df_bivariada.iterrows():
    sig = '✅' if row['significativa'] else '❌'
    print(f"{i+1:3d} {row['feature']:<45s} {row['tipo_teste']:<14s} "
          f"{row['efeito']:>8.4f} {row['magnitude']:<10s} "
          f"{row['p_value']:>12.2e} {sig:>5s}")

# Features com efeito >= Médio
bivariada_top = set(
    df_bivariada[df_bivariada['magnitude'].isin(['Médio', 'Grande'])]['feature']
)
# Features significativas
bivariada_sig = set(
    df_bivariada[df_bivariada['significativa']]['feature']
)

print(f"\n  Features com efeito Médio ou Grande: {len(bivariada_top)}")
print(f"  Features estatisticamente significativas: {len(bivariada_sig)}")

5.2 — Análise Bivariada: Associação com Churn

Correção de Bonferroni: alpha = 0.000926

──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  # Feature                                       Teste            Efeito Mag.            p-value  Sig.
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1 tempo_cliente_meses                           Mann-Whitney     0.1086 Médio          1.71e-80     ✅
  2 tipo_canal_Digital                            Qui-Quadrado     0.1024 Médio          6.71e-70     ✅
  3 tipo_canal_Corretor                           Qui-Quadrado     0.1008 Médio          7.77e-68     ✅
  4 cobertura_Premium                             Qui-Quadrado     0.0976 Pequeno        1.23e-63     ✅
  5 comissao_percentual                           Mann-Whitney     0.0924 Pequeno        3.05e-57     ✅
  6 cobertura_Básica                              Qui-Quadrado   

## 5.3 — Lasso (Regularização L1)

Lasso penaliza coeficientes, forçando os menos relevantes a zero.

Usamos LogisticRegression com penalty='l1' e variamos a força da regularização (C) para ver quais features persistem.

C pequeno = mais regularização (mais features zeradas)

C grande  = menos regularização (mais features mantidas)

Testamos múltiplos valores de C via cross-validation.

In [30]:
print(f"\n{'=' * 76}")
print("5.3 — Lasso (Regularização L1)")
print(f"{'=' * 76}")

# Cross-validation com múltiplos valores de C
print("\nAjustando Lasso com CV (pode levar alguns minutos)...")
lasso_cv = LogisticRegressionCV(
    Cs=50                    # 50 valores de C testados
    , penalty='l1'
    , solver='saga'          # suporta L1
    , cv=5                   # 5-fold CV
    , scoring='roc_auc'      # otimiza AUC
    , random_state=42
    , max_iter=5000
    , n_jobs=-1
    , class_weight='balanced' # compensa desbalanceamento
)
lasso_cv.fit(X_treino_scaled[COLS_MODELO], y_treino)

print(f"\n  Melhor C: {lasso_cv.C_[0]:.6f}")
print(f"  Melhor AUC (CV): {lasso_cv.scores_[1].mean(axis=0).max():.4f}")

# Coeficientes do melhor modelo
coefs_lasso = pd.DataFrame({
    'feature': COLS_MODELO
    , 'coef_lasso': lasso_cv.coef_[0]
    , 'abs_coef': np.abs(lasso_cv.coef_[0])
}).sort_values('abs_coef', ascending=False).reset_index(drop=True)

# Features selecionadas (coeficiente != 0)
lasso_selecionadas = set(coefs_lasso[coefs_lasso['abs_coef'] > 0]['feature'])
lasso_eliminadas = set(coefs_lasso[coefs_lasso['abs_coef'] == 0]['feature'])

print(f"\n{'─' * 76}")
print(f"{'#':>3s} {'Feature':<45s} {'Coef':>10s} {'Status':>10s}")
print(f"{'─' * 76}")
for i, row in coefs_lasso.iterrows():
    status = '✅' if row['abs_coef'] > 0 else '❌ ZERO'
    print(f"{i+1:3d} {row['feature']:<45s} {row['coef_lasso']:>10.4f} {status:>10s}")

print(f"\n  Features mantidas (coef ≠ 0): {len(lasso_selecionadas)}")
print(f"  Features eliminadas (coef = 0): {len(lasso_eliminadas)}")
if lasso_eliminadas:
    print(f"  Eliminadas: {sorted(lasso_eliminadas)}")


5.3 — Lasso (Regularização L1)

Ajustando Lasso com CV (pode levar alguns minutos)...

  Melhor C: 0.059636
  Melhor AUC (CV): 0.6110

────────────────────────────────────────────────────────────────────────────
  # Feature                                             Coef     Status
────────────────────────────────────────────────────────────────────────────
  1 cobertura_Premium                                -0.3718          ✅
  2 faixa_etaria_1) 18-25                             0.3414          ✅
  3 tipo_canal_Digital                                0.2427          ✅
  4 tipo_canal_Corretor                              -0.2290          ✅
  5 tempo_cliente_meses                              -0.2230          ✅
  6 cobertura_Básica                                  0.1973          ✅
  7 mes_inicio_11                                     0.0682          ✅
  8 mes_inicio_3                                      0.0605          ✅
  9 vigencia_meses                                   -0.0517  

## 5.4 — Critério de Informação (AIC/BIC) — Modelo Completo

Ajusta o modelo completo e avalia a significância individual (p-value) pelo teste de Wald. Features com p-value > 0.05 são candidatas a remoção.

Também reportamos AIC e BIC do modelo completo vs. modelo reduzido.

In [31]:
print(f"\n{'=' * 76}")
print("5.4 — Significância Individual (Wald) + AIC/BIC")
print(f"{'=' * 76}")

# Modelo completo
X_completo = sm.add_constant(X_treino_scaled[COLS_MODELO])
modelo_completo = sm.GLM(
    y_treino, X_completo, family=sm.families.Binomial()
).fit()

print(f"\n  AIC modelo completo: {modelo_completo.aic:.2f}")
print(f"  BIC modelo completo: {modelo_completo.bic:.2f}")

# Extrai p-values (teste de Wald)
pvalues = modelo_completo.pvalues.drop('const')
coefs = modelo_completo.params.drop('const')

df_wald = pd.DataFrame({
    'feature': pvalues.index
    , 'coef': coefs.values
    , 'p_value': pvalues.values
}).sort_values('p_value').reset_index(drop=True)

df_wald['significativa_05'] = df_wald['p_value'] < 0.05
df_wald['significativa_01'] = df_wald['p_value'] < 0.01

print(f"\n{'─' * 78}")
print(f"{'#':>3s} {'Feature':<45s} {'Coef':>8s} {'p-value':>12s} {'Sig.':>6s}")
print(f"{'─' * 78}")
for i, row in df_wald.iterrows():
    sig = '✅' if row['significativa_05'] else '❌'
    print(f"{i+1:3d} {row['feature']:<45s} {row['coef']:>8.4f} "
          f"{row['p_value']:>12.4e} {sig:>5s}")

wald_selecionadas = set(df_wald[df_wald['significativa_05']]['feature'])
wald_eliminadas = set(df_wald[~df_wald['significativa_05']]['feature'])

print(f"\n  Significativas (p < 0.05): {len(wald_selecionadas)}")
print(f"  Não significativas:        {len(wald_eliminadas)}")
if wald_eliminadas:
    print(f"  Candidatas a remoção: {sorted(wald_eliminadas)}")


5.4 — Significância Individual (Wald) + AIC/BIC

  AIC modelo completo: 40138.02
  BIC modelo completo: -266321.03

──────────────────────────────────────────────────────────────────────────────
  # Feature                                           Coef      p-value   Sig.
──────────────────────────────────────────────────────────────────────────────
  1 tempo_cliente_meses                            -0.2262   7.6607e-80     ✅
  2 cobertura_Premium                              -0.3515   1.2442e-19     ✅
  3 faixa_etaria_1) 18-25                           0.3593   1.2232e-14     ✅
  4 cobertura_Básica                                0.1719   2.2610e-08     ✅
  5 vigencia_meses                                 -0.0527   9.6945e-06     ✅
  6 franquia_media                                  0.1308   1.0249e-02     ✅
  7 ramo_Automóvel                                 -0.1833   3.0076e-02     ✅
  8 qtd_apolices_acumuladas                        -0.0407   4.6942e-02     ✅
  9 qtd_coberturas_dis

## 5.5 — Boruta (Wrapper com Random Forest)

Boruta cria "shadow features" (cópias embaralhadas de cada variável) e usa Random Forest para testar se cada feature real é significativamente melhor que a melhor shadow feature.

Resultado: Confirmed (aceita), Rejected (rejeitada), Tentative (incerta)

In [38]:
print(f"\n{'=' * 76}")
print("5.5 — Boruta (Wrapper com Random Forest)")
print(f"{'=' * 76}")

# Random Forest base para o Boruta
rf_boruta = RandomForestClassifier(
    n_estimators=200
    , max_depth=7
    , class_weight='balanced'
    , random_state=42
    , n_jobs=-1
)

# Boruta
boruta = BorutaPy(
    estimator=rf_boruta
    , n_estimators='auto'
    , max_iter=10            # máximo de iterações
    , random_state=42
    , verbose=0
)

print("\nExecutando Boruta (pode levar vários minutos)...")
boruta.fit(
    X_treino_scaled[COLS_MODELO].values
    , y_treino.values
)

# Resultados
boruta_ranking = pd.DataFrame({
    'feature': COLS_MODELO
    , 'ranking': boruta.ranking_
    , 'suporte': boruta.support_
    , 'suporte_fraco': boruta.support_weak_
})

boruta_ranking['decisao'] = boruta_ranking.apply(
    lambda r: 'Confirmed' if r['suporte']
    else ('Tentative' if r['suporte_fraco'] else 'Rejected')
    , axis=1
)
boruta_ranking = boruta_ranking.sort_values('ranking').reset_index(drop=True)

print(f"\n{'─' * 69}")
print(f"{'#':>3s} {'Feature':<45s} {'Rank':>8s} {'Decisão':<12s}")
print(f"{'─' * 69}")
for i, row in boruta_ranking.iterrows():
    emoji = '✅' if row['decisao'] == 'Confirmed' \
        else '🟡' if row['decisao'] == 'Tentative' else '❌'
    print(f"{i+1:3d} {row['feature']:<45s} {row['ranking']:>5d} "
          f"{emoji} {row['decisao']:<12s}")

boruta_confirmed = set(
    boruta_ranking[boruta_ranking['decisao'] == 'Confirmed']['feature']
)
boruta_tentative = set(
    boruta_ranking[boruta_ranking['decisao'] == 'Tentative']['feature']
)
boruta_rejected = set(
    boruta_ranking[boruta_ranking['decisao'] == 'Rejected']['feature']
)

print(f"\n  Confirmed: {len(boruta_confirmed)}")
print(f"  Tentative: {len(boruta_tentative)}")
print(f"  Rejected:  {len(boruta_rejected)}")


5.5 — Boruta (Wrapper com Random Forest)

Executando Boruta (pode levar vários minutos)...

─────────────────────────────────────────────────────────────────────
  # Feature                                           Rank Decisão     
─────────────────────────────────────────────────────────────────────
  1 tempo_cliente_meses                               1 ✅ Confirmed   
  2 comissao_percentual                               1 ✅ Confirmed   
  3 cobertura_Premium                                 1 ✅ Confirmed   
  4 tipo_canal_Digital                                1 ✅ Confirmed   
  5 tipo_canal_Corretor                               1 ✅ Confirmed   
  6 ratio_premio_vs_medio                             2 🟡 Tentative   
  7 cobertura_Básica                                  2 🟡 Tentative   
  8 franquia_media                                    3 ❌ Rejected    
  9 faixa_etaria_1) 18-25                             4 ❌ Rejected    
 10 nome_canal_Venda Digital (Site)                   5 

## 5.6 — Stepwise Bidirecional (Forward-First)

Adiciona features uma a uma, escolhendo a que mais melhora o AIC.

Para por quando nenhuma adição melhora o AIC.

Implementação manual com statsmodels (GLM logístico).

### Atenção, o procedimento completo é bastante demorado!

In [ ]:
print(f"\n{'=' * 76}")
print("5.6 — Stepwise Bidirecional (baseado em AIC)")
print(f"{'=' * 76}")

def stepwise_both_forward(X, y, verbose=True):
    """Stepwise bidirecional começando do vazio (forward-first)."""
    restantes = list(X.columns)
    selecionadas = []
    modelo_nulo = sm.GLM(y, sm.add_constant(pd.DataFrame({'c': np.ones(len(y))})),
                         family=sm.families.Binomial()).fit()
    melhor_aic = modelo_nulo.aic
    if verbose:
        print(f"  Modelo nulo — AIC: {melhor_aic:.2f}")
    passo = 0
    while True:
        passo += 1
        melhorou = False
        # FORWARD
        melhor_feat, melhor_aic_fwd = None, melhor_aic
        for feat in restantes:
            cands = selecionadas + [feat]
            try:
                m = sm.GLM(y, sm.add_constant(X[cands]), family=sm.families.Binomial()).fit()
                if m.aic < melhor_aic_fwd:
                    melhor_aic_fwd = m.aic
                    melhor_feat = feat
            except Exception:
                continue
        if melhor_feat:
            selecionadas.append(melhor_feat)
            restantes.remove(melhor_feat)
            delta = melhor_aic - melhor_aic_fwd
            melhor_aic = melhor_aic_fwd
            melhorou = True
            if verbose:
                print(f"  Passo {passo:2d} [+] {melhor_feat:<40s} AIC={melhor_aic:.2f} (Δ={delta:.2f})")
        # BACKWARD
        if len(selecionadas) >= 2:
            pior_feat, melhor_aic_bwd = None, melhor_aic
            for feat in selecionadas:
                cands = [f for f in selecionadas if f != feat]
                try:
                    m = sm.GLM(y, sm.add_constant(X[cands]), family=sm.families.Binomial()).fit()
                    if m.aic < melhor_aic_bwd:
                        melhor_aic_bwd = m.aic
                        pior_feat = feat
                except Exception:
                    continue
            if pior_feat:
                selecionadas.remove(pior_feat)
                restantes.append(pior_feat)
                delta = melhor_aic - melhor_aic_bwd
                melhor_aic = melhor_aic_bwd
                melhorou = True
                if verbose:
                    print(f"  Passo {passo:2d} [-] {pior_feat:<40s} AIC={melhor_aic:.2f} (Δ={delta:.2f})")
        if not melhorou:
            if verbose:
                print(f"  Passo {passo:2d}: Sem melhoria. Parada.")
            break
    return selecionadas, melhor_aic

print("\nExecutando forward-first...\n")
fwd_features, fwd_aic = stepwise_both_forward(X_treino_scaled[COLS_MODELO], y_treino, verbose=True)
fwd_set = set(fwd_features)
print(f"\n  Selecionadas (forward-first): {len(fwd_set)}")
print(f"  AIC: {fwd_aic:.2f}")


In [ ]:
# 5.6 Stepwise Forward-First
def stepwise(X, y, p_enter=0.05, p_leave=0.10):
    included, excluded = [], list(X.columns)
    for _ in range(50):
        changed = False
        p_vals = {}
        for c in excluded:
            try:
                m = sm.Logit(y, sm.add_constant(X[included + [c]])).fit(disp=0)
                p_vals[c] = m.pvalues[c]
            except: pass
        if not p_vals: break
        best = min(p_vals, key=p_vals.get)
        if p_vals[best] < p_enter:
            included.append(best)
            excluded.remove(best)
            changed = True
        elif any(m.pvalues.get(c, 1) > p_leave for c in included if c != 'const'):
            # Backward check inside loop (bidirectional)
            pass 
    return included
step_fw = stepwise(X_train[lasso_selected], y_train)
print(f"✅ 6.4 Stepwise Fwd: {len(step_fw)} features")


## 5.7 — Stepwise Bidirecional (Backward-First)

Adiciona features uma a uma, escolhendo a que mais melhora o AIC.

Para por quando nenhuma adição melhora o AIC.

Implementação manual com statsmodels (GLM logístico).

### Atenção, o procedimento completo é bastante demorado!

In [ ]:
# ────────────────────────────────────────────────────────
print(f"\n{'=' * 76}")
print("5.7 — Stepwise Bidirecional BACKWARD-FIRST (baseado em AIC)")
print(f"{'=' * 76}")

def stepwise_both_backward(X, y, verbose=True):
    """Stepwise bidirecional começando com todas as variáveis."""
    selecionadas = list(X.columns)
    restantes = []
    modelo_cheio = sm.GLM(y, sm.add_constant(X), family=sm.families.Binomial()).fit()
    melhor_aic = modelo_cheio.aic
    if verbose:
        print(f"  Modelo cheio ({len(selecionadas)} vars) — AIC: {melhor_aic:.2f}")
    passo = 0
    while True:
        passo += 1
        melhorou = False
        # BACKWARD
        if len(selecionadas) >= 2:
            pior_feat, melhor_aic_bwd = None, melhor_aic
            for feat in selecionadas:
                cands = [f for f in selecionadas if f != feat]
                try:
                    m = sm.GLM(y, sm.add_constant(X[cands]), family=sm.families.Binomial()).fit()
                    if m.aic < melhor_aic_bwd:
                        melhor_aic_bwd = m.aic
                        pior_feat = feat
                except Exception:
                    continue
            if pior_feat:
                selecionadas.remove(pior_feat)
                restantes.append(pior_feat)
                delta = melhor_aic - melhor_aic_bwd
                melhor_aic = melhor_aic_bwd
                melhorou = True
                if verbose:
                    print(f"  Passo {passo:2d} [-] {pior_feat:<40s} AIC={melhor_aic:.2f} (Δ={delta:.2f})")
        # FORWARD
        melhor_feat, melhor_aic_fwd = None, melhor_aic
        for feat in restantes:
            cands = selecionadas + [feat]
            try:
                m = sm.GLM(y, sm.add_constant(X[cands]), family=sm.families.Binomial()).fit()
                if m.aic < melhor_aic_fwd:
                    melhor_aic_fwd = m.aic
                    melhor_feat = feat
            except Exception:
                continue
        if melhor_feat:
            selecionadas.append(melhor_feat)
            restantes.remove(melhor_feat)
            delta = melhor_aic - melhor_aic_fwd
            melhor_aic = melhor_aic_fwd
            melhorou = True
            if verbose:
                print(f"  Passo {passo:2d} [+] {melhor_feat:<40s} AIC={melhor_aic:.2f} (Δ={delta:.2f})")
        if not melhorou:
            if verbose:
                print(f"  Passo {passo:2d}: Sem melhoria. Parada.")
            break
    return selecionadas, melhor_aic

print("\nExecutando backward-first...\n")
bwd_features, bwd_aic = stepwise_both_backward(X_treino_scaled[COLS_MODELO], y_treino, verbose=True)
bwd_set = set(bwd_features)
print(f"\n  Selecionadas (backward-first): {len(bwd_set)}")
print(f"  AIC: {bwd_aic:.2f}")

In [ ]:
# ── COMPARAÇÃO ──
print(f"\n{'=' * 76}")
print("5.7 — Comparação dos dois Stepwise")
print(f"{'=' * 76}")
print(f"  Forward-first:  {len(fwd_set)} vars, AIC={fwd_aic:.2f}")
print(f"  Backward-first: {len(bwd_set)} vars, AIC={bwd_aic:.2f}")
print(f"  Em comum:       {sorted(fwd_set & bwd_set)}")
print(f"  Só forward:     {sorted(fwd_set - bwd_set)}")
print(f"  Só backward:    {sorted(bwd_set - fwd_set)}")

## 5.8 — Matriz de Consenso

Para cada feature, verificamos quantos métodos a selecionaram.

Quanto mais métodos concordam, mais confiança na inclusão/exclusão.

Métodos considerados (5 votos possíveis):
  1. Bivariada (efeito Médio/Grande OU significativa com Bonferroni)
  2. Lasso (coeficiente ≠ 0)
  3. Stepwise Bidirecional (selecionada)
  4. Wald (p < 0.05 no modelo completo)
  5. Boruta (Confirmed)

VIF não "seleciona" features — ele alerta sobre colinearidade.

Usamos o VIF como critério de desempate/ajuste posterior.

In [ ]:
print(f"\n{'=' * 76}")
print("5.8 — Matriz de Consenso entre Métodos de Seleção")
print(f"{'=' * 76}")

todas_features = sorted(COLS_MODELO)

consenso_data = []
for col in todas_features:
    in_biv   = col in bivariada_sig
    in_vif   = col in vif_baixa
    in_lasso = col in lasso_selecionadas
    in_fwd   = col in fwd_set
    in_bwd   = col in bwd_set
    in_bor   = col in boruta_confirmed
    
    votos = sum([in_biv, in_vif, in_lasso, in_fwd, in_bwd, in_bor])
    
    consenso_data.append({
        'feature':  col,
        'bivariada': '✅' if in_biv   else '  ',
        'VIF':       '✅' if in_vif   else '  ',
        'lasso':     '✅' if in_lasso else '  ',
        'sw_fwd':    '✅' if in_fwd   else '  ',
        'sw_bwd':    '✅' if in_bwd   else '  ',
        'boruta':    '✅' if in_bor   else '  ',
        'votos':     votos
    })

consenso = pd.DataFrame(consenso_data).sort_values('votos', ascending=False)

# VIF como informação complementar
X_vif = sm.add_constant(X_treino_scaled[COLS_MODELO])
vif_map = {}
for i, col in enumerate(X_vif.columns):
    if col != 'const':
        try:
            vif_map[col] = variance_inflation_factor(X_vif.values, i)
        except Exception:
            vif_map[col] = np.nan
consenso['VIF_valor'] = consenso['feature'].map(vif_map).round(1)

# Ordena por votos (desc) e VIF (asc)
consenso = consenso.sort_values(
    ['votos', 'VIF'], ascending=[False, True]
).reset_index(drop=True)

# Exibe
print(f"\n{'─' * 110}")
print("5.8 — Matriz de Consenso entre Métodos de Seleção (6 votos)")
print(f"\n{'─' * 110}")
print(f"\n{'':>3s} {'Feature':<45s} {'Biv':>4s} {'VIF':>4s} {'Las':>4s} {'SWf':>4s} {'SWb':>4s} {'Bor':>4s} {'Votos':>6s} {'VIF':>8s}")
print(f"{'─' * 110}")
for _, row in consenso.iterrows():
    print(f"  {row['feature']:<45s} {row['bivariada']:>4s} {row['VIF']:>4s} "
          f"{row['lasso']:>4s} {row['sw_fwd']:>4s} {row['sw_bwd']:>4s} {row['boruta']:>4s} "
          f"{row['votos']:>4d}/6  {row['VIF_valor']:>7.1f}")

# Resumo por quantidade de votos
print(f"\n{'─' * 76}")
print("RESUMO POR QUANTIDADE DE VOTOS:")
print(f"{'─' * 76}")
for v in sorted(consenso['votos'].unique(), reverse=True):
    n = len(consenso[consenso['votos'] == v])
    feats = list(consenso[consenso['votos'] == v]['feature'])
    print(f"  {int(v)}/6 votos: {n} features")

## 5.9 — Proposta de Seleção Final

Critério de seleção:
  - 5/5 ou 4/5 votos: INCLUIR automaticamente
  - 3/5 votos: INCLUIR se VIF ≤ 10 (avaliar caso a caso)
  - 2/5 votos: EXCLUIR por default (incluir apenas se justificativa forte)
  - 1/5 ou 0/5 votos: EXCLUIR

Ajustes manuais por colinearidade:
  - Se nome_canal e tipo_canal ambos selecionados → manter nome_canal
    (mais granular) e remover tipo_canal
  - Se premio_mensal e premio_medio_mensal ambos selecionados → avaliar
    qual tem mais poder preditivo individual

In [ ]:
print(f"\n{'=' * 76}")
print("5.9 — Proposta de Seleção Final")
print(f"{'=' * 76}")

VARS_FINAL = []
avaliar_vif = []
removidas_colinearidade = []

for _, row in consenso.iterrows():
    col = row['feature']
    votos = row['votos']
    vif_val = row['VIF_valor']
    
    if votos >= 4:
        if vif_val <= 10:
            VARS_FINAL.append(col)
        else:
            removidas_colinearidade.append(col)
    elif votos == 3 and vif_val <= 10:
        VARS_FINAL.append(col)
        avaliar_vif.append(col)

print(f"\n  ✅ Incluídas (≥4 votos):           {sum(1 for _,r in consenso.iterrows() if r['votos'] >= 4 and r['VIF_valor'] <= 10)}")
print(f"  ✅ Incluídas (3 votos, VIF ≤ 10):  {len(avaliar_vif)}")
print(f"  ⚠️ Removidas (colinearidade):      {len(removidas_colinearidade)}")
print(f"  ❌ Excluídas (≤ 2 votos):          {sum(1 for _,r in consenso.iterrows() if r['votos'] <= 2)}")
print(f"\n  📋 VARIÁVEIS SELECIONADAS ({len(VARS_FINAL)}):")

for i, col in enumerate(sorted(VARS_FINAL), 1):
    votos = int(consenso[consenso['feature'] == col]['votos'].values[0])
    vif_val = consenso[consenso['feature'] == col]['VIF_valor'].values[0]
#     tipo = 'CONT' if col in COLS_CONTINUAS + COLS_TEMPORAIS_CONT \
    tipo = 'CONT' if col in COLS_CONTINUAS \
        else 'BIN' if col in COLS_BINARIAS else 'DUMMY'
    print(f"  {i:3d}. {col:<45s} [{tipo:>5s}] votos={votos}/6  VIF={vif_val:.1f}")

if avaliar_vif:
    print(f"\n  🟡 VARIÁVEIS PARA AVALIAÇÃO MANUAL (3 votos + VIF baixo):")
    for col in sorted(avaliar_vif):
        votos = int(consenso[consenso['feature'] == col]['votos'].values[0])
        vif_val = consenso[consenso['feature'] == col]['VIF_valor'].values[0]
        print(f"     - {col} (VIF={vif_val:.1f})")

if removidas_colinearidade:
    print(f"\n  ⚠️ REMOVIDAS POR COLINEARIDADE:")
    for col in sorted(removidas_colinearidade):
        print(f"     - {col}")

# Divergências entre os dois stepwise
so_fwd = fwd_set - bwd_set
so_bwd = bwd_set - fwd_set
if so_fwd or so_bwd:
    print(f"\n{'─' * 110}")
    print("  📊 DIVERGÊNCIAS ENTRE STEPWISE FORWARD-FIRST vs BACKWARD-FIRST:")
    if so_fwd:
        print(f"     Só no Forward-first:  {sorted(so_fwd)}")
    if so_bwd:
        print(f"     Só no Backward-first: {sorted(so_bwd)}")
else:
    print(f"\n  ℹ️ Os dois Stepwise selecionaram as mesmas variáveis.")

print(f"\n✅ Etapa 5 concluída.")
print(f"   Próximo: Etapa 6 (Padronização e Preparação Final).")

# Etapa 6 — Modelo Final de Regressão Logística

In [ ]:
print(f"  Variáveis finais: {sorted(VARS_FINAL)}")

## 6.1 — Ajuste do Modelo Final

In [ ]:
print(f"\n{'=' * 76}")
print("6.1 — Ajuste do Modelo Final (treino balanceado)")
print(f"{'=' * 76}")

scaler_final = StandardScaler()
X_modelo_scaled = pd.DataFrame(
    scaler_final.fit_transform(df_treino_bal[VARS_FINAL]),
    columns=VARS_FINAL, index=df_treino_bal.index
)
X_teste_scaled = pd.DataFrame(
    scaler_final.transform(df_teste[VARS_FINAL]),
    columns=VARS_FINAL, index=df_teste.index
)

X_fin_modelo = sm.add_constant(X_modelo_scaled)
X_fin_teste  = sm.add_constant(X_teste_scaled)
y_modelo = df_treino_bal[COL_TARGET]
y_teste  = df_teste[COL_TARGET]

print(f"  X modelo (balanceado): {X_fin_modelo.shape}  Churn: {y_modelo.mean():.2%}")
print(f"  X teste  (original):   {X_fin_teste.shape}  Churn: {y_teste.mean():.2%}")

modelo_final = sm.GLM(y_modelo, X_fin_modelo, family=sm.families.Binomial()).fit()
print(f"\n")
print(modelo_final.summary())

## 6.2 — Cross-Validation (5 folds)

In [ ]:
# ── 6.2 — Cross-Validation (5-Fold) no treino balanceado ──
print(f"\n{'=' * 76}")
print("6.2 — Cross-Validation (5-Fold Estratificado)")
print(f"{'=' * 76}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_metrics = {'auc': [], 'f1': [], 'precision': [], 'recall': [], 'accuracy': []}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_modelo_scaled[VARS_FINAL], y_modelo), 1):
    X_tr  = sm.add_constant(X_modelo_scaled[VARS_FINAL].iloc[tr_idx])
    X_val = sm.add_constant(X_modelo_scaled[VARS_FINAL].iloc[val_idx])
    y_tr  = y_modelo.iloc[tr_idx]
    y_val = y_modelo.iloc[val_idx]
    
    m = sm.GLM(y_tr, X_tr, family=sm.families.Binomial()).fit()
    y_prob = m.predict(X_val)
    y_pred = (y_prob >= 0.5).astype(int)
    
    cv_metrics['auc'].append(roc_auc_score(y_val, y_prob))
    cv_metrics['f1'].append(f1_score(y_val, y_pred))
    cv_metrics['precision'].append(precision_score(y_val, y_pred))
    cv_metrics['recall'].append(recall_score(y_val, y_pred))
    cv_metrics['accuracy'].append(accuracy_score(y_val, y_pred))
    
    print(f"  Fold {fold}: AUC={cv_metrics['auc'][-1]:.4f}  F1={cv_metrics['f1'][-1]:.4f}  "
          f"Prec={cv_metrics['precision'][-1]:.4f}  Rec={cv_metrics['recall'][-1]:.4f}")

print(f"\n  {'Média':>8s}: AUC={np.mean(cv_metrics['auc']):.4f}  F1={np.mean(cv_metrics['f1']):.4f}  "
      f"Prec={np.mean(cv_metrics['precision']):.4f}  Rec={np.mean(cv_metrics['recall']):.4f}")
print(f"  {'Std':>8s}: AUC={np.std(cv_metrics['auc']):.4f}  F1={np.std(cv_metrics['f1']):.4f}  "
      f"Prec={np.std(cv_metrics['precision']):.4f}  Rec={np.std(cv_metrics['recall']):.4f}")

## 6.3 — Métricas na Base de Teste

In [ ]:
print(f"\n{'=' * 76}")
print("6.3 — Métricas no Teste (proporção original)")
print(f"{'=' * 76}")

y_prob_teste = modelo_final.predict(X_fin_teste)

# Threshold otimizado por F1
thrs = np.arange(0.10, 0.90, 0.01)
f1s = [f1_score(y_teste, (y_prob_teste >= t).astype(int)) for t in thrs]
best_t = thrs[np.argmax(f1s)]
print(f"  Threshold ótimo (F1): {best_t:.2f} → F1={max(f1s):.4f}")

for t_label, t_val in [("Default (0.50)", 0.5), (f"Otimizado ({best_t:.2f})", best_t)]:
    y_pred = (y_prob_teste >= t_val).astype(int)
    cm = confusion_matrix(y_teste, y_pred)
    cm_pct = cm / cm.sum() * 100
    
    print(f"\n  ── Threshold: {t_label} ──")
    print(f"  Accuracy:  {accuracy_score(y_teste, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_teste, y_pred, zero_division=0):.4f}")
    print(f"  Recall:    {recall_score(y_teste, y_pred):.4f}")
    print(f"  F1-Score:  {f1_score(y_teste, y_pred):.4f}")
    print(f"  AUC-ROC:   {roc_auc_score(y_teste, y_prob_teste):.4f}")
    print(f"  Brier:     {brier_score_loss(y_teste, y_prob_teste):.4f}")
    
    labels = ['Ativo', 'Churn']
    print(f"\n  {'':>12s} {'Pred Ativo':>12s} {'Pred Churn':>12s}")
    for i, lab in enumerate(labels):
        print(f"  {lab:>12s} {cm[i,0]:>6d} ({cm_pct[i,0]:5.1f}%) {cm[i,1]:>6d} ({cm_pct[i,1]:5.1f}%)")

## 6.4 — Odds Ratio e Interpretação

In [ ]:
print(f"\n{'=' * 90}")
print("6.4 — Odds Ratios (exp(β))")
print(f"{'=' * 90}")

odds = pd.DataFrame({
    'coef': modelo_final.params,
    'odds_ratio': np.exp(modelo_final.params),
    'p_valor': modelo_final.pvalues,
    'IC_2.5%': np.exp(modelo_final.conf_int()[0]),
    'IC_97.5%': np.exp(modelo_final.conf_int()[1]),
})
odds['significancia'] = odds['p_valor'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else '.'))
)
odds = odds.sort_values('odds_ratio', ascending=False)
print(odds.to_string(float_format='%.4f'))

## 6.5 — Curva ROC

In [ ]:
print("\n" + "=" * 76)
print("6.5 — Curva ROC no Teste (proporção original)")
print("=" * 76)

fpr, tpr, _ = roc_curve(y_teste, y_prob_teste)
auc_val = roc_auc_score(y_teste, y_prob_teste)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'Modelo Final (AUC = {auc_val:.4f})')
plt.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Aleatório')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC — Modelo Final (teste com proporção original)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('curva_roc_modelo_final_undersampled.png', dpi=150)
plt.show()

## 6.6 — Teste de Kolmogorov-Smirnov (Capacidade de Separação)

In [ ]:
print(f"\n{'=' * 76}")
print("6.6 — Teste de Kolmogorov-Smirnov (Capacidade de Separação)")
print(f"{'=' * 76}")

# Scores por classe (base de teste)
scores_churn     = y_prob_teste[y_teste == 1]
scores_nao_churn = y_prob_teste[y_teste == 0]

# K-S via scipy.stats (já importado)
ks_stat, ks_pvalue = stats.ks_2samp(scores_churn, scores_nao_churn)

print(f"\n  K-S Statistic: {ks_stat:.4f}")
print(f"  p-value:       {ks_pvalue:.2e}")

# Interpretação
if ks_stat >= 0.50:
   faixa = "Excelente (≥0.50)"
elif ks_stat >= 0.40:
   faixa = "Muito Bom (0.40–0.50)"
elif ks_stat >= 0.30:
   faixa = "Bom (0.30–0.40)"
elif ks_stat >= 0.20:
   faixa = "Razoável (0.20–0.30)"
else:
   faixa = "Fraco (<0.20)"
print(f"  Classificação: {faixa}")

# Ponto de máxima separação
bins = np.linspace(0, 1, 1001)
cdf_churn     = np.searchsorted(np.sort(scores_churn), bins, side='right') / len(scores_churn)
cdf_nao_churn = np.searchsorted(np.sort(scores_nao_churn), bins, side='right') / len(scores_nao_churn)
idx_max = np.argmax(np.abs(cdf_churn - cdf_nao_churn))
print(f"  Ponto de máxima separação: score = {bins[idx_max]:.4f}")

# Gráfico das CDFs
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(bins, cdf_churn,     label='Churn (classe 1)',     color='crimson',   linewidth=2)
ax.plot(bins, cdf_nao_churn, label='Não-Churn (classe 0)', color='steelblue', linewidth=2)
ax.axvline(bins[idx_max], color='gray', linestyle='--', alpha=0.7,
           label=f'Máx. separação (score={bins[idx_max]:.3f})')
ax.annotate(f'K-S = {ks_stat:.4f}',
            xy=(bins[idx_max], (cdf_churn[idx_max] + cdf_nao_churn[idx_max]) / 2),
            fontsize=13, fontweight='bold', ha='left',
            xytext=(bins[idx_max] + 0.05, 0.5),
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_xlabel('Score (probabilidade prevista)', fontsize=12)
ax.set_ylabel('CDF acumulada', fontsize=12)
ax.set_title('Teste de Kolmogorov-Smirnov — Separação do Modelo', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6.7 — Teste de Hosmer-Lemeshow (Calibração)

In [ ]:
print(f"\n{'=' * 76}")
print("6.7 — Hosmer-Lemeshow + Calibração")
print(f"{'=' * 76}")

def hosmer_lemeshow(y_true, y_prob, g=10):
    df_hl = pd.DataFrame({'y': y_true, 'prob': y_prob})
    df_hl['grupo'] = pd.qcut(df_hl['prob'], g, duplicates='drop')
    obs = df_hl.groupby('grupo')['y'].sum()
    exp = df_hl.groupby('grupo')['prob'].sum()
    n = df_hl.groupby('grupo')['y'].count()
    hl_stat = (((obs - exp) ** 2) / (exp * (1 - exp / n))).sum()
    p_val = 1 - stats.chi2.cdf(hl_stat, g - 2)
    return hl_stat, p_val

hl_stat, hl_p = hosmer_lemeshow(y_teste.values, y_prob_teste.values)
print(f"  Estatística HL: {hl_stat:.4f}")
print(f"  p-valor:        {hl_p:.4f}")
print(f"  {'✅ Boa calibração' if hl_p > 0.05 else '⚠️ Calibração insuficiente'}")

frac_pos, mean_pred = calibration_curve(y_teste, y_prob_teste, n_bins=10)
plt.figure(figsize=(7, 5))
plt.plot(mean_pred, frac_pos, 's-', label='Modelo')
plt.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Calibração perfeita')
plt.xlabel('Probabilidade prevista')
plt.ylabel('Fração observada de churn')
plt.title('Curva de Calibração — Modelo Final')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('calibracao_modelo_final_undersampled.png', dpi=150)
plt.show()
print("  ✅ Gráfico salvo: calibracao_modelo_final_undersampled.png")

## 6.8 — Resumo Executivo do Modelo

In [ ]:
print(f"\n{'=' * 76}")
print("6.8 — Resumo Executivo")
print("=" * 76)
print(f"""
  Variáveis:              {len(VARS_FINAL)}
  Treino (balanceado):    {len(y_treino):,} (50/50)
  Teste (original):       {len(y_teste):,} ({y_teste.mean():.1%} churn)
  AIC:                    {modelo_final.aic:.2f}
  AUC-ROC (teste):        {auc_val:.4f}
  AUC-ROC (CV média):     {np.mean(cv_metrics['auc']):.4f} ± {np.std(cv_metrics['auc']):.4f}
  Pseudo-R² (McFadden):   {1 - modelo_final.llf / modelo_final.llnull:.4f}
  Threshold ótimo:        {best_t:.2f}
""")

print("  ✅ Etapa 6 concluída.")

### O modelo estatístico confirma que os drivers de churn são canal, cobertura, faixa etária e tempo de relacionamento. A capacidade preditiva limitada (AUC=0.61) indica que fatores não capturados nos dados — como experiência de atendimento, preço da concorrência e dados de sinistralidade — também são relevantes, reforçando a necessidade de capturar essas informações em versões futuras deste modelo.

# Etapa 7 - Exportação dos Resultados

## 7.1 - Tabela de Resumo da Fórmula para o Power BI

In [ ]:
tModeloRegressao = pd.DataFrame({
    'variavel':         modelo_final.params.index,
    'coeficiente_b':    modelo_final.params.values,
    'odds_ratio':       np.exp(modelo_final.params.values),
    'p_valor':          modelo_final.pvalues.values,
    'intervalo_inf':    np.exp(modelo_final.conf_int()[0].values),
    'intervalo_sup':    np.exp(modelo_final.conf_int()[1].values),
    'delta_odds_ratio': np.exp(modelo_final.params.values) - 1,
    'delta_int_inf':    np.exp(modelo_final.conf_int()[0].values) - 1,
    'delta_int_sup':    np.exp(modelo_final.conf_int()[1].values) - 1
})

tModeloRegressao = tModeloRegressao[tModeloRegressao['variavel'] != 'const'].reset_index(drop=True)

tModeloRegressao['significancia'] = tModeloRegressao['p_valor'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else '.'))
)

tModeloRegressao.to_csv('C:\\Users\\bruno\\repos\\mba_xperiun_repos\\mba_xperiun_dc4\\Dados\\tModeloRegressaoUndersampled.csv', index=False, sep=';', decimal=',')
print(tModeloRegressao.to_string(index=False))
print("\n✅ Exportado: tModeloRegressaoUndersampled.csv")

## 7.2 - Tabela de Decis para o Power BI

In [ ]:
# Usa base de TESTE para evitar overfitting visual
df_decil = pd.DataFrame({
    'prob_churn': modelo_final.predict(X_fin_teste),
    'churn_real': y_teste.values
})

df_decil['decil'] = pd.qcut(df_decil['prob_churn'], 10, labels=[f'D{i}' for i in range(1, 11)])

resumo_decil = df_decil.groupby('decil', observed=True).agg(
    qtd=('churn_real', 'count'),
    churns=('churn_real', 'sum')
).reset_index()

resumo_decil['taxa_churn'] = (resumo_decil['churns'] / resumo_decil['qtd'])
taxa_global = df_decil['churn_real'].mean()

# Tabela para exportar ao Power BI
resumo_decil['taxa_churn_global'] = taxa_global
resumo_decil['lift_vs_global'] = (resumo_decil['taxa_churn'] / taxa_global)
resumo_decil['acum_churns'] = resumo_decil['churns'].cumsum()
resumo_decil['pct_churns_capturados'] = (resumo_decil['acum_churns'] / resumo_decil['churns'].sum() * 100)

resumo_decil.to_csv('C:\\Users\\bruno\\repos\\mba_xperiun_repos\\mba_xperiun_dc4\\Dados\\tDecisModeloUndersampled.csv', index=False, sep=';', decimal=',')
print("\n" + resumo_decil.to_string(index=False))
print(f"\n  Taxa global de churn: {100 * taxa_global:.2f}%")
print(f"  Lift D10 vs D1: {resumo_decil['taxa_churn'].iloc[-1] / resumo_decil['taxa_churn'].iloc[0]:.2f}x")
print(f"\n✅ Exportados: grafico_decis_modelo.png + tDecisModeloUndersampled.csv")

# Gráfico
fig, ax = plt.subplots(figsize=(10, 6))
cores = ['#2E7D32' if t < taxa_global else '#C62828' for t in resumo_decil['taxa_churn']]
bars = ax.bar(resumo_decil['decil'], resumo_decil['taxa_churn'], color=cores, edgecolor='white')
ax.axhline(y=taxa_global, color='#333', linestyle='--', linewidth=1.2, label=f'Taxa global: {taxa_global:.1f}%')

# Rótulos nas barras
for bar, taxa in zip(bars, resumo_decil['taxa_churn']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{taxa:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Decil de Risco (D1 = menor risco → D10 = maior risco)')
ax.set_ylabel('Taxa de Churn Real (%)')
ax.set_title('Gráfico de Decis — Poder de Ordenação do Modelo')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('grafico_decis_modelo_undersampled.png', dpi=150)
plt.show()

## 7.3 — Recuperando os parâmetros usados no StandardScaler

In [ ]:
print(f"\n{'=' * 76}")
print("7.3 — Recuperando os parâmetros usados no StandardScaler")
print("=" * 76)

# Assumindo que o scaler é um StandardScaler e foi fitado em X_treino
# Ajuste o nome do scaler se necessário (ex: scaler, std_scaler, etc.)

params_scaler = pd.DataFrame({
    'feature': scaler_final.feature_names_in_,
    'media':   scaler_final.mean_,
    'desvio':  scaler_final.scale_
})

# Filtra só as contínuas de interesse
continuas = ['tempo_cliente_meses', 'vigencia_meses', 'premio_medio_mensal']
print(params_scaler[params_scaler['feature'].isin(continuas)].to_string(index=False))

# Para reverter manualmente: valor_original = (valor_scaled * desvio) + media